# 🔍 Local RAG Pipeline

A local Retrieval-Augmented Generation (RAG) pipeline using:
- **ChromaDB** for vector storage
- **SentenceTransformers** (`all-MiniLM-L6-v2`) for embeddings
- **Groq / LLaMA 3** as the language model
- **BM25** for keyword-based pre-filtering (HyDE technique)
- **pdfplumber** for PDF text extraction

## 1. Import Packages

In [1]:
import os
import re

import pandas as pd
import requests
from pprint import pprint
from dotenv import dotenv_values

from groq import Groq
from google import genai
from google.genai import types
from openai import OpenAI

import chromadb
from chromadb.utils import embedding_functions
from sentence_transformers import SentenceTransformer

import pdfplumber
from pypdf import PdfReader
from rank_bm25 import BM25Okapi
from sklearn.metrics.pairwise import cosine_similarity

from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

/home/abdellah/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Configuration

Load API keys and secrets from the `.env` file.

In [2]:
# Load environment variables
config = dotenv_values(".env")

GROQ_API_KEY  = config["GROQ_API_KEY"]
GEMINI_API_KEY = config["GEMINI_API_KEY"]
HF_TOKEN      = config.get("HF_TOKEN")

# Make HuggingFace token available to the transformers library
os.environ["HF_TOKEN"] = HF_TOKEN

## 3. Embedding Model

Load `all-MiniLM-L6-v2` from SentenceTransformers. This model converts text into 384-dimensional vectors.

In [3]:
model = SentenceTransformer("all-MiniLM-L6-v2")

# Quick sanity check
sample_sentence = "What is the Meaning of Life?"
sample_embedding = model.encode(sample_sentence)
print(f"Embedding shape: {sample_embedding.shape}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1520.72it/s]


Embedding shape: (384,)


## 4. Vector Database (ChromaDB)

Initialize a persistent ChromaDB collection. Documents will be stored on disk under `./course/`.

In [4]:
chroma_client    = chromadb.PersistentClient(path="course")
collection_name  = "course"
embedding_fn     = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

collection = chroma_client.get_or_create_collection(
    name=collection_name,
    embedding_function=embedding_fn
)
print(collection)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3162.80it/s]


Collection(name=course)


## 5. PDF Text Extraction

Read a PDF file page by page, clean the extracted text, and return a list of page documents.

In [5]:
def get_text_from_pdf(path):
    documents = []
    with pdfplumber.open(path) as pdf:
        print(f"Number of pages: {len(pdf.pages)}")

        for i, page in enumerate(pdf.pages):
            text = page.extract_text()

            if text:
                # Remove leading/trailing whitespace
                text = text.strip()

                # Fix soft line breaks inside paragraphs
                text = re.sub(r'(?<!\n)\n(?!\n)', ' ', text)

                # Normalize multiple spaces / newlines
                text = re.sub(r'\s+', ' ', text)

                documents.append({"id": i, "text": text})
            else:
                documents.append("")

    return documents


documents = get_text_from_pdf("xml_course.pdf")
print(f"Total pages loaded: {len(documents)}")

Number of pages: 55
Total pages loaded: 55


## 6. Text Chunking

Split each page's text into overlapping fixed-size chunks so that long pages fit within embedding/LLM context limits.

In [6]:
def split_text(text, chunk_size=1000, chunk_overlap=30):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - chunk_overlap
    return chunks

In [7]:
# Preview the first document
documents[0]

{'id': 0,
 'text': 'XML : Cours et exercices corrigés Master Web Intelligence et Sciences des Données (WISD) Prof. El Habib NFAOUI (elhabib.nfaoui@usmba.ac.ma) Edition 2020'}

In [8]:
# Split all documents into chunks
chunked_documents = []
for doc in documents:
    chunks = split_text(doc["text"])
    for i, chunk in enumerate(chunks):
        chunked_documents.append({
            "id"  : f"{doc['id']}_chunk{i + 1}",
            "text": chunk
        })

print(f"Total chunks: {len(chunked_documents)}")

Total chunks: 93


## 7. Embedding Generation

Encode each text chunk into a dense vector using the SentenceTransformer model.

In [9]:
def get_embedding(text):
    """Return the embedding vector for a given text string."""
    embeddings = model.encode(text)
    return embeddings

In [10]:
# Generate and attach embeddings to every chunk
for doc_chunk in chunked_documents:
    doc_chunk["embeddings"] = get_embedding(doc_chunk["text"])

## 8. Index Chunks into ChromaDB

Upsert all chunks (text + embedding) into the persistent vector store.

In [11]:
for doc_chunk in chunked_documents:
    collection.upsert(
        ids       =[doc_chunk["id"]],
        documents =[doc_chunk["text"]],
        embeddings=[doc_chunk["embeddings"]]
    )

print("Indexing complete.")

Indexing complete.


## 9. LLM Integration (Groq / LLaMA 3)

Use the Groq API with `llama-3.1-8b-instant` to generate a hypothetical document for a user question (HyDE — Hypothetical Document Embeddings).

In [12]:
client_groq = Groq(api_key=GROQ_API_KEY)

def get_llm_documents(question):
    """Generate a short hypothetical documentation passage for `question`."""
    completion = client_groq.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {
                "role"   : "system",
                "content": (
                    f"You are a technical documentation writer. "
                    f"Write one clear, structured documentation for: {question}. "
                    f"Use simple English words, be brief, and stay on topic."
                ),
            }
        ],
        temperature=1,
        max_completion_tokens=1024,
        top_p=1,
        stream=True,
        stop=None
    )

    res = [chunk.choices[0].delta.content for chunk in completion]
    res = [s for s in res if s]
    return "".join(res)

In [13]:
# Test HyDE document generation
response = get_llm_documents(question="What is AI?")
print(response)

**What is Artificial Intelligence (AI)?**

**Table of Contents**

1. [Introduction to AI](#introduction-to-ai)
2. [How AI Works](#how-ai-works)
3. [Types of AI](#types-of-ai)
4. [Benefits and Applications of AI](#benefits-and-applications-of-ai)
5. [Conclusion](#conclusion)

**Introduction to AI**

Artificial Intelligence (AI) refers to the creation of computer systems that can think, learn, and act like humans. AI systems can perform tasks that typically require human intelligence, such as recognizing images, understanding natural language, and solving complex problems.

**How AI Works**

AI works by using algorithms and data to make decisions and predictions. There are two main types of AI:

1. **Narrow AI**: Also known as Weak AI, this type of AI is designed to perform a specific task, such as facial recognition or language translation.
2. **General AI**: Also known as Strong AI, this type of AI is designed to perform any intellectual task that a human can, such as reasoning, proble

## 10. LLM Document Splitting & Embedding

Break the hypothetical LLM response into paragraph-level chunks and embed them.

In [14]:
def split_text_llm(text):
    """Split LLM-generated text by newlines, keeping only substantive paragraphs."""
    chunks = []
    for i, paragraph in enumerate(text.split("\n")):
        paragraph = paragraph.strip()
        if len(paragraph) > 50:
            chunks.append({
                "id"       : f"chunk_{i}",
                "text"     : paragraph,
                "embedding": None
            })
    return chunks

In [15]:
def get_llm_embedding(response):
    """Split and embed an LLM-generated response."""
    llm_documents = split_text_llm(response)
    for doc_chunk in llm_documents:
        doc_chunk["embeddings"] = get_embedding(doc_chunk["text"])
    return llm_documents

## 11. BM25 Keyword Search

`collection.query()` does semantic (dense) search. Here we add a BM25 (sparse) keyword pass to pre-filter candidates before re-ranking by cosine similarity.

In [16]:
def bm25_search(bm25, question, chunked_documents, top_k=2):
    """
    Return the top-k chunks most relevant to `question` using BM25 scoring.

    Parameters
    ----------
    bm25             : BM25Okapi  — pre-built BM25 index
    question         : str        — user query
    chunked_documents: list[dict] — original chunk list (text + embeddings)
    top_k            : int        — number of results to return
    """
    tokenized_query = question.split()
    scores          = bm25.get_scores(tokenized_query)
    top_indices     = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]

    return [
        {
            "text"      : chunked_documents[i]["text"],
            "embeddings": chunked_documents[i]["embeddings"],
            "score"     : scores[i]
        }
        for i in top_indices
    ]

## 12. Full Retrieval Pipeline (HyDE + BM25)

Combine HyDE (Hypothetical Document Embeddings) with BM25 keyword filtering:

1. Generate a hypothetical answer with the LLM  
2. Embed the hypothetical answer  
3. Use BM25 to filter the most keyword-relevant corpus chunks

In [17]:
question = "utilisé pour une partie du cours intitulé XML ?"

# Step 1 — Generate a hypothetical answer (HyDE)
hyde_doc = get_llm_documents(question)

# Step 2 — Embed the hypothetical answer
hyde_doc_embedding = get_llm_embedding(hyde_doc)
print("HyDE embeddings:", hyde_doc_embedding)

# Step 3 — BM25 pre-filtering on the corpus
corpus = [doc["text"].split() for doc in chunked_documents]
bm25   = BM25Okapi(corpus)
filtered_chunks = bm25_search(bm25, question=question, chunked_documents=chunked_documents)
print("\nBM25 top chunks:", filtered_chunks)

HyDE embeddings: [{'id': 'chunk_0', 'text': '**Titre : Utilisation de XML pour stocker des données**', 'embedding': None, 'embeddings': array([-4.77838814e-02,  8.35794806e-02,  1.39875459e-02, -6.30554408e-02,
       -5.22101820e-02, -1.40753612e-02,  4.29463796e-02,  7.74639845e-02,
        5.70297241e-02, -1.49473157e-02,  6.70735091e-02, -3.47220153e-02,
        4.25524674e-02, -4.81044129e-02,  2.68982886e-03,  2.79723462e-02,
       -1.18887834e-02, -2.06993558e-04,  1.66561361e-02,  4.87189516e-02,
       -2.79551875e-02, -2.03288589e-02, -8.52912366e-02,  2.15863041e-03,
        1.33013893e-02,  3.21940854e-02, -5.32141067e-02,  2.38576774e-02,
        4.02735211e-02, -1.85907017e-02,  1.61806829e-02,  7.33232722e-02,
        2.12991778e-02,  7.55228773e-02, -2.89584398e-02,  1.54481195e-02,
       -4.83712647e-03, -1.17214061e-01, -5.47246225e-02,  1.14826344e-01,
       -7.40967169e-02,  2.90572681e-02, -8.74509215e-02, -6.04299940e-02,
       -4.71076332e-02, -5.90362726e-03

In [ ]:
def rerank_with_bm25(hyde_embeddings, bm25_chunks, top_k=2):
    """
        Compare the reel filtred chunks that generated by BM25
        with the embeddings hypothetical HyDE.
        return chunks classed by similarity.
    """
    # Extract Vectors HyDE  (shape: [n_hyde, 384])
    hyde_vecs = np.array([doc["embeddings"] for doc in hyde_embeddings])

    results = []
    for chunk in bm25_chunks:
        chunk_vec = np.array(chunk["embeddings"]).reshape(1, -1)

        # Cosine similarity with with this chunk and each vecor in hyde vectors
        scores = cosine_similarity(chunk_vec, hyde_vecs)
        best_score = float(scores.max())  # get max score

        results.append({
            "text" : chunk["text"],
            "score": best_score
        })

    # Sort by Score
    results.sort(key=lambda x: x["score"], reverse=True)
    return results[:top_k]

In [ ]:
def rerank_with_sematic():
    pass

In [ ]:
# Re-Rank with hybrid search
'''
    - hybrid search with AI responses by combining keyword search with semantic.
    - which looks for exact words) and semantic search (which looks for meaning).
'''
def hybrid_search(question, chunked_documents, collection, top_k=5, k_rrf=60):
    
    # ── Keyword Search by BM25 ─────────────────────────────────────────
    corpus      = [doc["text"].split() for doc in chunked_documents]
    bm25        = BM25Okapi(corpus)
    bm25_scores = bm25.get_scores(question.split())
    bm25_ranked = sorted(range(len(bm25_scores)),
                         key=lambda i: bm25_scores[i], reverse=True)

    # ── Semantic Search (ChromaDB) ─────────────────────────────────────
    chroma_results = collection.query(query_texts=[question], n_results=top_k)
    chroma_ids     = chroma_results["ids"][0]

    # ── Merge the Two (RRF) ────────────────────────────────────────────
    rrf_scores = {}

    for rank, idx in enumerate(bm25_ranked[:top_k]):
        doc_id = chunked_documents[idx]["id"]
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0) + 1 / (k_rrf + rank + 1)

    for rank, doc_id in enumerate(chroma_ids):
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0) + 1 / (k_rrf + rank + 1)

    sorted_ids = sorted(rrf_scores, key=rrf_scores.get, reverse=True)[:top_k]

    # ── Return Result as Chunks With Embedding ─────────────────────────
    id_to_chunk = {doc["id"]: doc for doc in chunked_documents}
    return [id_to_chunk[doc_id] for doc_id in sorted_ids if doc_id in id_to_chunk]

In [29]:
chunked_documents

[{'id': '0_chunk1',
  'text': 'XML : Cours et exercices corrigés Master Web Intelligence et Sciences des Données (WISD) Prof. El Habib NFAOUI (elhabib.nfaoui@usmba.ac.ma) Edition 2020',
  'embeddings': array([-4.90652435e-02,  3.07165831e-02, -5.08692004e-02, -3.80176399e-03,
          8.05748776e-02, -9.01683196e-02, -1.11428397e-02, -2.38312855e-02,
          2.54876981e-03, -1.61064770e-02,  5.69283590e-02, -1.90612003e-02,
          1.26303792e-01, -8.51056501e-02, -2.56653056e-02,  1.07253686e-01,
         -2.41060965e-02, -3.79411504e-02, -1.06827654e-02, -1.02055266e-01,
          8.06717128e-02,  9.09017324e-02, -3.09490482e-03, -1.17884696e-01,
         -3.18723544e-02,  6.42051995e-02, -1.21293981e-02, -1.12362057e-02,
          9.90836602e-03, -1.95025969e-02,  2.54168902e-02,  4.78931032e-02,
          3.69620174e-02,  7.11256564e-02, -3.62732485e-02,  2.80075222e-02,
         -3.83577868e-02, -9.19257030e-02,  5.15829995e-02,  1.23736739e-01,
         -3.91160958e-02,  2.1

In [ ]:
def get_context_from_chunks(relevant_chunks):
    context = [c["text"] for c in relevant_chunks]
    return " ".join(context)

In [20]:
# Function That Generate Reponse
client_groq = Groq(api_key=GROQ_API_KEY)
def generate_reponse(question, context):
    sys_prompt = f"""
        You are an assistant for question-answering tasks. Use the following pieces of 
        retrieved context to answer the question. If you don't know the answer, say that you
        don't know. Use three sentences maximum and keep the answer concise.
            Instructions:
            - Be helpful and answer questions concisely. If you don't know the answer, say 'I don't know'
            - Utilize the context provided for accurate and specific information.
            - Incorporate your preexisting knowledge to enhance the depth and relevance of your response.
            - Cite your sources
        Context :{context}
        """
    completion = client_groq.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
        {
            "role": "system",
            "content": sys_prompt,
        },
        {
            "role": "user",
            "content": question
        }
        ],
        temperature=1,
        max_completion_tokens=1024,
        top_p=1,
        stream=True,
        stop=None
    )

    res = []
    for chunk in completion:
        res.append(chunk.choices[0].delta.content)

    res = [s for s in res if s]
    res = "".join(res)
    return res

In [32]:
question    = "What is XML used for?"
hyde_doc    = get_llm_documents(question)
hyde_embeds = get_llm_embedding(hyde_doc)

relevant_chunks = hybrid_search(question, chunked_documents, collection)
print(relevant_chunks)

# Re-ranking cosine and context
# context = get_context_from_chunks(relevant_chunks=relevant_chunks)

# Get Final Reponse
# response = generate_reponse(question=question, context=context)
# print(response)

[{'id': '4_chunk1', 'text': "Chapitre 1 Structure des documents XML 1. Introduction XML est un méta-langage universel pour représenter les données échangées sur le Web qui permet au développeur de délivrer du contenu depuis les applications à d'autres applications ou aux navigateurs. XML fournit simplement une structure fondamentale et un ensemble de règles qui doivent être respectées par tous les langages de balisage. XML standardise la manière dont l'information est : - échangée - présentée - archivée - retrouvée - transformée - ... 2. Structure d’un document XML Un document XML est un document texte lisible. Un document XML est découpé en éléments structurés hiérarchiquement. Un document a un élément racine appelé élément du document. \uf070 Un élément est composé : ◼ d’un nom qui spécifie son type, ◼ d’attributs, ◼ d’un contenu formé d’éléments ou de textes. <producteur> <adresse> <rue>Azilal</rue> <ville>Casa</ville> 5", 'embeddings': array([-5.14884330e-02,  4.63688411e-02,  1.32